In [ ]:
#final code
# Enhanced Arabic Text Augmentation with Optimal Output

# Step 1: Install required packages
!pip install -q cohere nltk pandas tqdm
!pip install -q git+https://github.com/linuxscout/pyarabic.git
!wget -q https://raw.githubusercontent.com/mohataher/arabic-stop-words/master/list.txt -O arabic_stopwords.txt

# Step 2: Import libraries
import cohere
import nltk
import re
import time
import random
import pandas as pd
from pyarabic.araby import strip_tashkeel
from google.colab import files
from tqdm import tqdm
from datetime import timedelta

# Step 3: Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('perluniprops', quiet=True)

# Initialize with progress bar
tqdm.pandas()

# Custom Arabic tokenizer
def arabic_word_tokenize(text):
    return re.findall(r'[\w\u0600-\u06FF]+', text)

# Load Arabic stopwords
with open('arabic_stopwords.txt', 'r', encoding='utf-8') as f:
    arabic_stopwords = set(line.strip() for line in f)

# Initialize Cohere client
COHERE_API_KEY = "*****************"  # Replace with your API key
co = cohere.Client(COHERE_API_KEY)

# Enhanced rate limiting system
class RateLimiter:
    def __init__(self, calls_per_minute=8):
        self.calls_per_minute = calls_per_minute
        self.min_interval = 60 / calls_per_minute
        self.last_call_time = time.time()
        self.total_calls = 0

    def wait_if_needed(self):
        elapsed = time.time() - self.last_call_time
        if elapsed < self.min_interval:
            wait_time = self.min_interval - elapsed
            time.sleep(wait_time)
        self.last_call_time = time.time()
        self.total_calls += 1

limiter = RateLimiter(calls_per_minute=9)  # Slightly more aggressive

# Statistics tracker
class AugmentationStats:
    def __init__(self):
        self.start_time = time.time()
        self.total_texts = 0
        self.total_variations = 0
        self.words_replaced = 0
        self.api_calls = 0
        self.failed_calls = 0

    def get_elapsed(self):
        return timedelta(seconds=int(time.time() - self.start_time))

    def print_status(self):
        print(f"\nCurrent Status [Elapsed: {self.get_elapsed()}]")
        print(f"Texts: {self.total_texts} | Variations: {self.total_variations}")
        print(f"API Calls: {self.api_calls} | Failures: {self.failed_calls}")
        print(f"Avg Variations/Text: {self.total_variations/max(1,self.total_texts):.1f}")
        print(f"Avg Words Replaced: {self.words_replaced/max(1,self.total_variations):.1f}")

stats = AugmentationStats()

def clean_arabic_text(text):
    text = strip_tashkeel(text)
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\s]+', ' ', text)
    return text.strip()

def get_arabic_synonyms(word, max_retries=3):
    """Fetch 10 synonyms per word with enhanced prompt"""
    retries = 0
    while retries < max_retries:
        try:
            limiter.wait_if_needed()
            stats.api_calls += 1

            response = co.chat(
                model="command-r",
                message=f"""
                أعطني 10 مرادفات للكلمة العربية '{word}' مع مراعاة:
                1. أن تكون المرادفات صحيحة لغويًا
                2. أن تحافظ على معنى الكلمة الأصلي
                3. أن تكون مفصولة بفواصل فقط
                4. بدون أي شرح أو نص إضافي
                """,
                temperature=0.7,  # Slightly higher for more diversity
                max_tokens=200  # Increased for longer responses
            )

            # Enhanced cleaning of synonyms
            synonyms = [
                s.strip().replace('.', '').replace('،', '')
                for s in re.split(r'[,،\n]', response.text)
                if s.strip() and s.strip() != word
            ]
            return list(dict.fromkeys(synonyms))[:10]  # Remove duplicates while preserving order

        except Exception as e:
            stats.failed_calls += 1
            if "429" in str(e):
                wait = min(60 * (retries + 1), 300)
                print(f"Rate limited. Waiting {wait} seconds...")
                time.sleep(wait)
            retries += 1
    return []

def generate_variations(original_text, target_variations=5):
    """Generate richer variations with optimized replacement strategy"""
    tokens = arabic_word_tokenize(clean_arabic_text(original_text))
    synonyms_cache = {}

    # Smart synonym collection - prioritize content words
    for token in tokens:
        if (len(token) > 2 and  # Longer words tend to be more meaningful
            token not in arabic_stopwords and
            re.match(r'^[\u0600-\u06FF]+$', token)):
            synonyms_cache[token] = get_arabic_synonyms(token)

    variations = []
    attempts = 0
    max_attempts = target_variations * 2  # Allow some generation attempts

    while len(variations) < target_variations and attempts < max_attempts:
        attempts += 1
        new_tokens = []
        replacements = 0

        for token in tokens:
            if token in synonyms_cache and synonyms_cache[token] and random.random() < 0.7:
                new_tokens.append(random.choice(synonyms_cache[token]))
                replacements += 1
            else:
                new_tokens.append(token)

        # Only keep sufficiently different variations
        variation = ' '.join(new_tokens)
        variation = re.sub(r'\sال(\w)', r' \1', variation).strip()

        # Ensure minimum changes and no duplicates
        if (replacements >= max(1, len(tokens)//3) and  # At least 1/3 of words changed
            not any(v == variation for v, _ in variations)):
            variations.append((variation, replacements))

    return variations[:target_variations]  # Return only the requested number

# File processing with enhanced feedback
def process_file(input_file, output_file):
    df = pd.read_csv(input_file)
    stats.total_texts = len(df)

    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing Texts"):
        variations = generate_variations(row['text'], target_variations=random.randint(4, 6))

        for i, (variation, replacements) in enumerate(variations, 1):
            results.append({
                'id': row['id'],
                'technique': row['technique'],
                'variation_num': i,
                'original_text': row['text'],
                'augmented_text': variation,
                'words_replaced': replacements,
                'replacement_ratio': f"{(replacements/len(variation.split())):.1%}"
            })
            stats.total_variations += 1
            stats.words_replaced += replacements

        if random.random() < 0.2:  # Occasional status updates
            stats.print_status()

    output_df = pd.DataFrame(results)
    output_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    return output_df

# Main execution
print("Please upload your CSV file:")
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
output_filename = 'enhanced_augmented_sub_data.csv'

print(f"\nStarting augmentation at {time.ctime()}")
result_df = process_file(input_filename, output_filename)

# Final report
stats.print_status()
print(f"\nGenerated {len(result_df)} variations from {stats.total_texts} texts")
print(f"Success rate: {(stats.api_calls - stats.failed_calls)/stats.api_calls:.1%}")

# Download results
files.download(output_filename)
print("\nAugmentation complete! File downloaded:", output_filename)



  Preparing metadata (setup.py) ... done
Please upload your CSV file:


Saving sub_data_train_extracted_data_part4.csv to sub_data_train_extracted_data_part4.csv

Starting augmentation at Thu Jun 12 13:45:14 2025


Processing Texts:   3%|▎         | 4/120 [02:40<58:57, 30.49s/it]  


Current Status [Elapsed: 0:02:58]
Texts: 120 | Variations: 19
API Calls: 25 | Failures: 0
Avg Variations/Text: 0.2
Avg Words Replaced: 5.3


Processing Texts:  10%|█         | 12/120 [06:13<52:25, 29.13s/it]

...

Processing Texts: 100%|██████████| 120/120 [45:53<00:00, 22.95s/it]


Current Status [Elapsed: 0:46:11]
Texts: 120 | Variations: 581
API Calls: 414 | Failures: 0
Avg Variations/Text: 4.8
Avg Words Replaced: 2.7

Current Status [Elapsed: 0:46:11]
Texts: 120 | Variations: 581
API Calls: 414 | Failures: 0
Avg Variations/Text: 4.8
Avg Words Replaced: 2.7

Generated 581 variations from 120 texts
Success rate: 100.0%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Augmentation complete! File downloaded: enhanced_augmented_sub_data.csv
